## Import

In [164]:
import pandas as pd
import numpy as np
import json
import psycopg2
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
from dotenv import load_dotenv
import os
import ast

## Récupération des données / POSTES

In [165]:
load_dotenv()

USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = 'localhost'
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")


#variable qui stock l'url d'accès a notre db / sécurisé grace aux infos au dessus dans le .env
DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

try:

    #établie la connexion avec la db, comme un lancement de moteur genre
    engine = create_engine(DATABASE_URL)
    
    #ce que ca va aller chercher dans la db
    query = "SELECT id_poste, id_entreprise, titre, description, skills, annee_experience, salaire_min, salaire_max, type_contrat, remote_policy, langues, latitude, longitude FROM poste;"

    #grace a engine, on peut envoyer ma requete query aller chercher les éléments dans la db avec read_sql , panda permet qeu ce soit lisible en python
    df_postes = pd.read_sql(query, engine)
    
    print("Connexion réussie")
    

    fichier = 'dataset_postes.csv'

    df_postes.to_csv(fichier , index=False, encoding='utf-8')



except Exception as e:
    print(e)



Connexion réussie


## Récupération des données / USER

In [166]:
try:
    
    engine = create_engine(DATABASE_URL)

    query = """
        SELECT 
            u.id_utilisateur,
            u.nom,
            u.prenom,
            u.age,
            u.role,
            u.latitude_secteur,
            u.longitude_secteur,

            pu.id_profil,
            pu.titre_profession,
            pu.description,
            pu.experiences_annees,
            pu.diplomes,
            pu.hard_skills,
            pu.soft_skills,
            pu.langues,
            pu.jobs_passes,
            pu.candidatures_positives,
            pu.candidatures_refusees

        FROM utilisateur u
        JOIN profil_utilisateur pu ON u.id_utilisateur = pu.id_utilisateur
        WHERE u.role = 'candidat'
    """

    df_combined = pd.read_sql(query, engine)
    
    print("fichier csv créé")

    fichier = 'dataset_combined_users.csv'
    final = df_combined.to_csv(fichier, index=False, encoding='utf-8')

except Exception as e:
    print(f"Erreur : {e}")

fichier csv créé


## Préprocessing des données / Texte pour each user et poste

In [167]:
from sklearn.feature_extraction.text import TfidfVectorizer

postes = pd.read_csv("dataset_postes.csv", quotechar='"', on_bad_lines='skip') # quand il y a des ' ca évite de fiare des bugs
user = pd.read_csv("dataset_combined_users.csv")

vectorizer = TfidfVectorizer(stop_words=None) 


# CLEAN SKILLS CAR JSON LOUCHES

def clean_skills(x):
    try:
        skills_list = ast.literal_eval(x) # transforme en liste python lisible sinon crash 
        return ' '.join([k["name"] for k in skills_list]) # en mode pour chaque élement dans name , genre si ya name:javascript ca prend que javascript car faut pas tt prendre sinon ca va fausser le modele.
    except:
        return ''
    
postes["skills_clean"] = postes['skills'].apply(clean_skills)


# display(postes["skills_clean"])
# display(postes['skills'])




# CLEAN text job

postes["text"] = (  postes["titre"].fillna("") +  " "  +  postes["description"].fillna("") +  " "  + postes["skills_clean"].fillna("")  )



# CLEAN profils 

def clean_list(tontext):
    try:
        return ' '.join(ast.literal_eval(tontext))
    except:
        return ""
    
user["hard_skills_clean"] = user["hard_skills"].apply(clean_list) # j'utilsie la fonction sur ces deux la car en vrai cest les plus utiles pour les recommandation mais je verrais si je peux pas appliquer sur d'autres pour améliorer le modele
user["jobs_clean"] = user["jobs_passes"].apply(clean_list)




# CLEAN text user 

user["text"] = ( user["titre_profession"].fillna("") + " " + user["hard_skills_clean"].fillna("") + " " + user["jobs_clean"].fillna("") )

print (postes["text"])
print(user["text"])



0      Senior Back-End Developer Sensefuel\nFondée en...
1      Ingénieur .net / SQL confirmé(e) - Lead - H/F ...
2      Développeur.se  Full stack Java Concepteur-dév...
3      Principal Software Engineer Python en startup ...
4      Lead Software Engineer Fullstack (Node & Vue) ...
                             ...                        
240    DevOps AWS - Permanent - Full Remote Since 201...
241    Senior Mobile Engineer - Paris At Swile, we be...
242    Fullstack Engineer - NodeJS À propos de la squ...
243    Senior developer Full-Stack Ruby on Rails Nous...
244    Senior Mobile Engineer - Montpellier At Swile,...
Name: text, Length: 245, dtype: str
0    Développeuse Full Stack Python React PostgreSQ...
1    Data Analyst SQL Power BI Python Excel Tableau...
2    Chargée de Marketing Digital SEO Google Ads Ca...
3    Responsable Logistique SAP Gestion des stocks ...
4    Développeuse Mobile Flutter Dart Swift Firebas...
Name: text, dtype: str


## Modèle de prédiction 1 / Vectorization 

In [ ]:
user_vectorize = vectorizer.fit_transform(user['text'])
postes_vectorize = vectorizer.fit_transform(postes['text'])

print(user_vectorize)
print(postes_vectorize)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 75 stored elements and shape (5, 68)>
  Coords	Values
  (0, 25)	0.40459348095413744
  (0, 32)	0.25074159553115655
  (0, 59)	0.25074159553115655
  (0, 50)	0.20229674047706872
  (0, 51)	0.25074159553115655
  (0, 48)	0.25074159553115655
  (0, 23)	0.25074159553115655
  (0, 53)	0.25074159553115655
  (0, 5)	0.25074159553115655
  (0, 7)	0.25074159553115655
  (0, 60)	0.25074159553115655
  (0, 26)	0.25074159553115655
  (0, 38)	0.20229674047706872
  (0, 41)	0.25074159553115655
  (0, 28)	0.25074159553115655
  (1, 50)	0.1735796426265571
  (1, 19)	0.43029498588331905
  (1, 2)	0.43029498588331905
  (1, 58)	0.21514749294165952
  (1, 49)	0.21514749294165952
  (1, 8)	0.43029498588331905
  (1, 29)	0.1735796426265571
  (1, 64)	0.21514749294165952
  (1, 3)	0.21514749294165952
  (1, 36)	0.21514749294165952
  :	:
  (3, 55)	0.2297015380425108
  (3, 33)	0.2297015380425108
  (3, 21)	0.2297015380425108
  (3, 61)	0.2297015380425108
  (3, 62)	0.2297015